# Transformer 位置编码 · 第 2 讲
## Learned Positional Embedding：让模型自己学习每个位置的表示

> 学习路线：**回顾 Sinusoidal → 查表原理 → 手动创建位置向量 → `nn.Embedding` 封装 → 反向传播 → 影响 Attention → 长度限制 → 对比与思考**。

前一讲我们学习了 **固定 Sinusoidal Positional Encoding**，它根据 `pos` 通过 `sin/cos` 计算位置向量。本讲研究一个简单但重要的问题：

> 既然每个位置最终只是一个 $d_{model}$ 维向量，为什么不把这些向量当成参数，直接交给模型训练？

**本讲目标**

1. 从形状和查表操作理解 `nn.Embedding(max_len, d_model)`；
2. 区分 **Token Embedding** 和 **Position Embedding** 的索引方式；
3. 从梯度验证：只有本次用到的位置参数会被直接更新；
4. 知道可学习位置编码仍然在 **Q/K/V 之前** 与词向量相加；
5. 理解它为何**没有显式的相对距离公式**，以及超过预设位置范围的限制。

**运行环境**：Python + PyTorch；可选 `matplotlib`（用于最后的可视化）。每个代码单元格可按顺序独立执行，无需依赖上一讲的 Notebook。

## 0. 先把上一讲和本讲连起来

对于句子：

```text
Token：    苹果      很       甜
位置 pos：  0       1       2
```

上一讲使用固定公式构造位置向量：

$$
PE_{pos,2i}=\sin\left(\frac{pos}{10000^{2i/d_{model}}}\right),\quad
PE_{pos,2i+1}=\cos\left(\frac{pos}{10000^{2i/d_{model}}}\right).
$$

**本讲不再使用任何三角函数**。我们准备一个可学习矩阵 $P$，并按照位置编号取出对应的行：

$$
\boxed{P\in\mathbb{R}^{L_{max}\times d_{model}}},\qquad
\boxed{p_i=P[i,:]}.
$$

模型输入仍为：

$$
\boxed{H_{b,i,:}=X_{b,i,:}+p_i}.
$$

- $L_{max}$：预设的最大位置数量；
- $d_{model}$：每个位置向量的维度，要与词嵌入维度一致；
- $i$：位置下标，通常从 $0$ 开始；
- $P$：**训练过程中会变化**的位置参数矩阵。

因此，两种绝对位置编码的**注入位置相同**，主要区别在于：位置向量从固定函数计算，还是从可训练的参数表读取。

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
print('PyTorch 版本:', torch.__version__)

PyTorch 版本: 2.13.0+cu130


## 1. 暂时不用 `nn.Embedding`：先手动构造位置表

为了看清最本质的查表过程，我们假设：

- `max_len = 5`：位置只有 `0, 1, 2, 3, 4`；
- `d_model = 4`：每个位置用一个 4 维向量表示；
- 下面人为填入简单数字，**只为展示索引机制**，不是实际训练好的权重。

$$
P=\begin{bmatrix}
p_0^\top \\
p_1^\top \\
p_2^\top \\
p_3^\top \\
p_4^\top
\end{bmatrix}\in\mathbb{R}^{5\times 4}.
$$

查找位置 2 的向量就是 `P[2]`；它**不是**把数字 2 当作浮点特征输入网络，而是用数字 2 **索引矩阵的第 3 行**。

In [2]:
# 行对应“位置索引”，列对应“特征维度”。
position_table = torch.tensor([
    [0.1, 0.2, 0.3, 0.4],  # 位置 0
    [0.2, 0.1, 0.4, 0.3],  # 位置 1
    [0.5, 0.6, 0.7, 0.8],  # 位置 2
    [0.6, 0.5, 0.8, 0.7],  # 位置 3
    [0.9, 1.0, 1.1, 1.2],  # 位置 4
], dtype=torch.float32)

position_ids = torch.tensor([0, 1, 2])  # “苹果 很 甜”在序列中的位置

# 根据 position_ids 查表：输出形状是 (L=3, D=4)
pos_vectors = position_table[position_ids]

print('完整的位置参数表 shape:', tuple(position_table.shape))
print('位置 ID:', position_ids.tolist())
print('取出的位置向量 shape:', tuple(pos_vectors.shape))
print('本次位置向量:', pos_vectors)

assert torch.equal(pos_vectors[2], position_table[2])

完整的位置参数表 shape: (5, 4)
位置 ID: [0, 1, 2]
取出的位置向量 shape: (3, 4)
本次位置向量: tensor([[0.1000, 0.2000, 0.3000, 0.4000],
        [0.2000, 0.1000, 0.4000, 0.3000],
        [0.5000, 0.6000, 0.7000, 0.8000]])


### 1.1 Token Embedding 和 Position Embedding 都是查表，但查的“编号”不同

```text
Token：         苹果      很       甜
Token ID：       7       2       9  ──→ 词嵌入表 E_token[token_id]
Position ID：    0       1       2  ──→ 位置嵌入表 E_pos[position_id]
                        │
                        ▼
        H = Token Embedding + Position Embedding
```

两种 Embedding 都可以由 `nn.Embedding` 实现：

- `nn.Embedding(vocab_size, d_model)`：按**词表 ID**取行。同一个 Token（在相同词表下）取出的词向量相同；
- `nn.Embedding(max_len, d_model)`：按**序列位置 ID**取行。同一个位置对应相同的位置向量。

它们都是可训练参数，但分别回答不同问题：**是什么词？处于哪个位置？**

In [ ]:
# nn.Embedding 的本质是拥有一个可学习的 weight 参数矩阵。
max_len = 6
d_model = 8
# 自动保存为parameter
learned_pe = nn.Embedding(num_embeddings=max_len, embedding_dim=d_model)

print('位置表权重形状:', tuple(learned_pe.weight.shape))
print('weight 是 nn.Parameter？', isinstance(learned_pe.weight, nn.Parameter))
print('是否参与梯度计算？', learned_pe.weight.requires_grad)
print('可训练参数总量:', sum(p.numel() for p in learned_pe.parameters()))

# 输入 position ID (L,)；输出位置向量 (L,D)
pos_ids = torch.arange(3)  # [0, 1, 2]
pos_embeds = learned_pe(pos_ids)

print('位置 ID shape:', tuple(pos_ids.shape))
print('位置向量 shape:', tuple(pos_embeds.shape))

# 验证 nn.Embedding 的前向传播就是根据整数 ID 查表。
assert torch.equal(pos_embeds, learned_pe.weight[pos_ids])
assert pos_embeds.shape == (3, d_model)

位置表权重形状: (6, 8)
weight 是 nn.Parameter？ True
是否参与梯度计算？ True
可训练参数总量: 48
位置 ID shape: (3,)
位置向量 shape: (3, 8)


## 2. 加到 Token Embedding 上：具体的 Tensor 形状

设 Batch 大小为 $B$，序列长度为 $L$，隐藏维度为 $D$：

$$
X\in\mathbb{R}^{B\times L\times D},\quad
P_{selected}\in\mathbb{R}^{L\times D}.
$$

为了加到每个样本上，要把位置向量扩展成 `(1, L, D)`，利用 **broadcasting（广播）**：

$$
(B,L,D)+(1,L,D)\longrightarrow(B,L,D).
$$

注意：**不同样本可以共用同一组绝对位置 ID**；但如果数据存在不同起点或特殊填充策略，需明确构造相应 `position_ids`。这里先学习最标准的连续位置 `0,1,2,...`。

In [ ]:
# 和上一讲一样，用“苹果 很 甜”作演示。
word_to_id = {'苹果': 0, '很': 1, '甜': 2}
token_ids = torch.tensor([[0, 1, 2], [2, 1, 0]])  # (B=2, L=3)

token_embedding = nn.Embedding(num_embeddings=3, embedding_dim=d_model)

# 原始 Transformer：词嵌入可乘 sqrt(d_model)，然后加 PE。
# 本讲使用与原论文同一条主线，聚焦位置编码的结构。
word_vectors = token_embedding(token_ids) * math.sqrt(d_model)  # (2,3,8)
position_ids = torch.arange(token_ids.size(1))                 # (3,)
position_vectors = learned_pe(position_ids).unsqueeze(0)        # (1,3,8)

h = word_vectors + position_vectors  # (2,3,8)

print('Token IDs:', tuple(token_ids.shape))
print('Token Embedding:', tuple(word_vectors.shape))
print('Position Embedding:', tuple(position_vectors.shape))
print('相加后的 Transformer 输入:', tuple(h.shape))

assert h.shape == word_vectors.shape
assert torch.allclose(h[0, 0] - word_vectors[0, 0], learned_pe.weight[0])
assert torch.allclose(h[1, 0] - word_vectors[1, 0], learned_pe.weight[0])

## 3. 封装成可复用模块：`LearnedPositionalEncoding`

前面的写法虽然足够直观，但实际放入模型时，更希望由模块自动根据当前输入长度构造 `position_ids`。

我们实现以下接口：

```python
pos_encoding(x, start_pos=0)
```

- `x`：已经算好的词向量，形状 `(B,L,D)`；
- `start_pos`：本次输入的**第一个 Token 在完整序列中的绝对位置**；
- 返回：加上学习到的位置向量，形状不变。

`Dropout` 在这里保留为可选参数；设成 `0.0` 便于观察确定性的数值。**这不是完整 Transformer**，只是它的输入位置编码模块。

In [ ]:
class LearnedPositionalEncoding(nn.Module):
    """可学习的绝对位置编码：P[pos] 直接由 nn.Embedding 查表得到。"""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.0):
        super().__init__()

        if d_model <= 0 or max_len <= 0:
            raise ValueError('d_model 与 max_len 必须为正整数。')

        # 这样子出了__init__()，就可以在 forward() 里直接使用 self.d_model、self.max_len。
        # 没有self接收的话，函数结束后就会被销毁，forward()里就无法访问了。
        self.d_model = d_model
        self.max_len = max_len

        # 可训练参数：权重 shape = (max_len, d_model)
        # 与 Sinusoidal 的 register_buffer 不同，这里梯度可以更新 weight。
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        """x: (B,L,D)；start_pos: 本次第一个 Token 的绝对位置。"""
        if x.ndim != 3:
            raise ValueError('x 应为 (B, L, D) 三维张量。')
        if x.size(-1) != self.d_model:
            raise ValueError('x 的最后一维必须等于 d_model。')

        seq_len = x.size(1)
        if start_pos < 0 or start_pos + seq_len > self.max_len:
            raise ValueError('当前 Token 的位置超出了预设的 max_len。')

        # 例如 L=2、start_pos=3 → [3,4]，绝不能默认从 0 重新开始。
        # 推理的时候用的
        position_ids = torch.arange(
            start_pos, start_pos + seq_len, device=x.device, dtype=torch.long
        )

        # (L,D)，来自可学习的位置表。
        position_vectors = self.position_embedding(position_ids)

        # 将位置向量变成 (1,L,D)，与任意 B 的输入广播相加。
        position_vectors = position_vectors.unsqueeze(0).to(dtype=x.dtype)

        return self.dropout(x + position_vectors)


pos_encoding = LearnedPositionalEncoding(d_model=8, max_len=128)
example_x = torch.zeros(2, 3, 8)  # 故意设零，便于单独观察位置向量
example_h = pos_encoding(example_x)

print('输出形状:', tuple(example_h.shape))
print('参数名称与形状:')
for name, param in pos_encoding.named_parameters():
    print(' ', name, tuple(param.shape))

# 同一个位置向量应用于不同样本。assert 失败了才打印
assert torch.allclose(example_h[0], example_h[1])
assert sum(p.numel() for p in pos_encoding.parameters()) == 128 * 8

输出形状: (2, 3, 8)
参数名称与形状:
  position_embedding.weight (128, 8)


## 4. 为什么称为 **Learned**？用反向传播检查参数变化

仅仅看到 `requires_grad=True` 还不够。我们亲自做一次 `backward()`，观察：

1. 位置表中哪些行得到梯度；
2. 优化器执行 `step()` 后哪些行的值改变。

为保证现象直观，这个实验只查位置 `[0,1,2]`，**不使用其它位置**，也暂时不涉及 Transformer 的训练目标。

> 下例用普通 `nn.Embedding`，未启用 weight decay 等会更新未使用参数的优化器设置。对于 `nn.Embedding` 的默认密集梯度，被索引到的行有直接梯度，其余行梯度为零。

In [ ]:
torch.manual_seed(7)
grad_embedding = nn.Embedding(num_embeddings=6, embedding_dim=4)
optimizer = torch.optim.SGD(grad_embedding.parameters(), lr=0.5)

used_positions = torch.tensor([0, 1, 2])
old_weights = grad_embedding.weight.detach().clone()

# 简单的演示损失，目的是确保本次查询的三个向量参与梯度计算。
selected_vectors = grad_embedding(used_positions)  # (3, 4)
loss = selected_vectors.square().mean()

optimizer.zero_grad()
loss.backward()

# 检查每一行的梯度范数。
grad_norms = grad_embedding.weight.grad.norm(dim=1)
print('每个位置对应的梯度范数：', grad_norms.tolist())

assert torch.all(grad_norms[:3] > 0)
assert torch.all(grad_norms[3:] == 0)

# 梯度下降更新参数。
optimizer.step()
weight_changes = (grad_embedding.weight.detach() - old_weights).norm(dim=1)
print('优化一步后每个位置参数的变化量：', weight_changes.tolist())

assert torch.all(weight_changes[:3] > 0)
assert torch.all(weight_changes[3:] == 0)

print('结论：本次查到的位置 0,1,2 被更新；未查到的 3,4,5 未变化。')

### 4.1 一次反向传播究竟做了什么？

设某个位置的查表结果为：

$$p_i=P[i,:].$$

本轮损失为 $\mathcal L$，经过 Attention、FFN 或其他后续网络后反向传播到输入位置向量：

$$\frac{\partial\mathcal L}{\partial p_i}
\quad\Longrightarrow\quad
\frac{\partial\mathcal L}{\partial P[i,:]}.$$

优化器于是可以修改 $P[i,:]$。**没有人为指定“第 2 个位置应该是什么数字”**，位置向量的数值是为降低任务损失而学出来的。

这里有一个重要区别：

- **查表是固定的操作**（ID 0 就读第 0 行），它本身不学习如何索引；
- **表里的每一行是可学习的参数**，随着训练变化。

因此“可学习位置编码”不等于“模型通过公式推导位置”，而是“让任务决定每个位置向量的数值”。

## 5. 一个完整小训练：让模型学会区分不同位置

接着做一个极简的监督学习实验：把“位置 0、1、2、3”当作分类标签，输入都是同一个零词向量，唯一可用的信息来自位置向量。

**这是教学实验，不是语言模型 benchmark。** 它只证明位置参数可以与网络参数一起通过梯度学习，并不意味着真实模型会简单地记住位置分类。

In [ ]:
torch.manual_seed(11)
num_positions = 4
demo_d_model = 8

# 模型用同一组位置 ID，分别读取可学习的位置向量。
toy_pos_embedding = nn.Embedding(num_positions, demo_d_model)
toy_classifier = nn.Linear(demo_d_model, num_positions)

optimizer = torch.optim.Adam(
    list(toy_pos_embedding.parameters()) + list(toy_classifier.parameters()),
    lr=0.05
)

position_ids = torch.arange(num_positions)  # [0,1,2,3]
labels = position_ids                       # 监督标签恰好为位置编号

loss_history = []
for step in range(100):
    # 输入 (4,) -> 位置特征 (4,8) -> 分类得分 (4,4)
    position_features = toy_pos_embedding(position_ids)
    logits = toy_classifier(position_features)

    # 每个位置的正确类别就是它自己。
    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

with torch.no_grad():
    final_logits = toy_classifier(toy_pos_embedding(position_ids))
    predicted_positions = final_logits.argmax(dim=-1)

print('初始 loss: %.4f' % loss_history[0])
print('最终 loss: %.4f' % loss_history[-1])
print('预测位置:', predicted_positions.tolist())
print('实际位置:', labels.tolist())

assert loss_history[-1] < loss_history[0]
assert torch.equal(predicted_positions, labels)

## 6. 位置编码何时进入 Self-Attention？

**可学习绝对位置编码不是直接加在 Attention Score 上**；它和 Sinusoidal 一样，通常先加入输入表示，再经过 $W_Q,W_K,W_V$：

$$
H_i=x_i+p_i,
$$

$$
q_i=H_iW_Q,\quad k_i=H_iW_K,\quad v_i=H_iW_V.
$$

于是注意力得分含有：

$$
e_{ij}=\frac{\big((x_i+p_i)W_Q\big)\big((x_j+p_j)W_K\big)^\top}{\sqrt{d_k}}.
$$

你可以把它理解为：**Token 内容与绝对位置共同影响 Q/K/V，再影响注意力权重与输出**。

注意，位置 $i$ 和位置 $j$ 的距离 $i-j$ **没有被显式写进上式**；与下一讲的相对位置编码相比，这就是重要差别。训练后的参数可能学出有用的相对位置行为，但编码方式本身没有预先规定这种结构。

In [ ]:
def simple_self_attention(x: torch.Tensor):
    """教学用：Q=K=V=X，无因果 Mask，观察位置编码影响。"""
    q = k = v = x
    scores = q @ k.transpose(-2, -1) / math.sqrt(x.size(-1))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights


torch.manual_seed(21)
d_model = 8
word_table = nn.Embedding(3, d_model)
position_module = LearnedPositionalEncoding(d_model, max_len=8)

ids_a = torch.tensor([[0, 1, 2]])  # 苹果 很 甜
ids_b = torch.tensor([[2, 1, 0]])  # 甜 很 苹果
perm = torch.tensor([2, 1, 0])

# 为避免演示时大幅度数值使 softmax 接近饱和，此处不乘 sqrt(d_model)。
# 这个设定只用于比较“交换顺序”时的数学性质；不是完整 Transformer 前向。
x_a = word_table(ids_a)
x_b = word_table(ids_b)

without_pos_a, _ = simple_self_attention(x_a)
without_pos_b, _ = simple_self_attention(x_b)

with_pos_a, _ = simple_self_attention(position_module(x_a))
with_pos_b, _ = simple_self_attention(position_module(x_b))

print('不加位置编码：仅重排输出？',
      torch.allclose(without_pos_b, without_pos_a[:, perm, :], atol=1e-6))
print('加入可学习位置编码：仍是简单重排？',
      torch.allclose(with_pos_b, with_pos_a[:, perm, :], atol=1e-6))

# 相同 Token “苹果”分别在序列 A 的位置 0 与 B 的位置 2。
assert torch.allclose(x_a[0, 0], x_b[0, 2])
assert not torch.allclose(position_module(x_a)[0, 0], position_module(x_b)[0, 2])

print('相同的 Token Embedding 换位置后，加入的位置向量不同。')

# 严谨说明：本例的新位置表仍是随机初始化的，尚未经过真实语言训练。
# 它只能展示位置“参与了运算”，不能据此断言模型已经理解语序。

## 7. 与 Sinusoidal 对照：参数和结构区别在哪里？

| 对比项 | Sinusoidal PE | Learned Absolute PE |
|---|---|---|
| 位置向量的来源 | 根据 `sin/cos` 固定公式计算 | 按 `position_ids` 查可训练的表 |
| 位置专属可训练参数 | 0 | $L_{max}\times d_{model}$ |
| 是否能随着任务训练变化 | 不会直接更新 PE | 会通过反向传播更新 |
| 是否预设位置间的数学关系 | 有，多频率三角结构 | 没有固定的跨位置关系约束 |
| 对未训练过的更远位置 | 数学公式可继续计算，但模型不保证泛化 | 标准查表模型通常无法访问超出表范围的位置 |
| 一般注入位置 | Token Embedding 上相加 | Token Embedding 上相加 |

**一定要避免的误解：**

- Sinusoidal “公式可以算更长位置” $\ne$ “模型一定能处理更长上下文”；
- Learned “可训练” $\ne$ “在所有任务上比 Sinusoidal 更好”；
- 原始 Transformer 的 Table 3(E) 对比中，固定 Sinusoidal 和 Learned 的结果**非常接近**。选择固定函数主要是出于对长度外推的考虑。

### 7.1 看一眼位置向量的结构：随机初始化 vs 固定三角函数

下面画的是 **尚未训练的 Learned PE** 和 **Sinusoidal PE**。随机初始化的位置向量不应表现出正弦编码那样的规则波形；**但这不意味着训练后的位置向量一定仍然无规律**。

为了直观比较，我们用两个独立的热力图。

In [ ]:
def build_sinusoidal_table(max_len: int, d_model: int) -> torch.Tensor:
    """独立构造固定 sin/cos PE；此处要求 d_model 是偶数。"""
    if d_model % 2 != 0:
        raise ValueError('此教学实现要求 d_model 为偶数。')

    positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    pair_ids = torch.arange(d_model // 2, dtype=torch.float32)
    inv_freq = torch.exp(-math.log(10000.0) * (2 * pair_ids / d_model))
    angles = positions * inv_freq

    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe


visual_len, visual_d = 32, 16
sinusoidal_table = build_sinusoidal_table(visual_len, visual_d)
random_learned_table = nn.Embedding(visual_len, visual_d).weight.detach()

plt.figure(figsize=(8, 4))
plt.imshow(sinusoidal_table.T.numpy(), aspect='auto', origin='lower')
plt.xlabel('Position')
plt.ylabel('Embedding dimension')
plt.title('Sinusoidal PE (fixed, structured)')
plt.colorbar(label='Value')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.imshow(random_learned_table.T.numpy(), aspect='auto', origin='lower')
plt.xlabel('Position')
plt.ylabel('Embedding dimension')
plt.title('Learned Absolute PE (random initialization)')
plt.colorbar(label='Value')
plt.tight_layout()
plt.show()

## 8. 一个非常实际的问题：长上下文与 `start_pos`

如果训练或推理时输入新 Token，它仍然需要**完整序列中的绝对位置编号**。

假设已经有 3 个 Token：

```text
完整序列： 苹果   很   甜   [新1]  [新2]
位置 ID：  0    1    2      3     4
```

在使用历史 KV Cache 的自回归推理中，如果本次只新输入 `[新1, 新2]`，它们的绝对位置仍然应该是 `[3,4]`，而不是 `[0,1]`。这与上一讲的 `start_pos` 是同一原则。

但 Learned Absolute PE 还有一个问题：**如果位置 ID 超出了创建嵌入表时的 `max_len-1`，查表就没有对应的行。**

In [ ]:
# 故意使用零输入，输出直接等于位置向量，便于检验索引。
pos_module = LearnedPositionalEncoding(d_model=8, max_len=8)
new_x = torch.zeros(1, 2, 8)

# 正确的绝对位置：[3,4]。
correct_h = pos_module(new_x, start_pos=3)

# 错误示例：假装新增 Token 又从位置 0 开始。
wrong_h = pos_module(new_x, start_pos=0)

assert torch.allclose(correct_h[0, 0], pos_module.position_embedding.weight[3])
assert torch.allclose(correct_h[0, 1], pos_module.position_embedding.weight[4])
assert not torch.allclose(correct_h, wrong_h)

print('正确索引：位置 3、4')
print('错误索引：位置 0、1')
print('两种结果是否相同？', torch.allclose(correct_h, wrong_h))

# max_len=8，合法位置 ID 为 0~7，取到位置 8 时会越界。
try:
    pos_module(torch.zeros(1, 1, 8), start_pos=8)
except ValueError as exc:
    print('越界测试（预期报错）：', exc)

### 8.1 能不能把 `max_len` 设得非常大？

可以，但这并不是免费的长度外推。

例如 `d_model=768`、`max_len=2048`，位置编码表需要：

$$2048\times768=1,572,864\ \text{个可训练参数}.$$

如果只见过前 512 个位置，那么第 513 个以后的行即使**在表里存在**，也可能根本没得到有效训练。

因此要区分两种情况：

1. **超过表的上限**：标准 `nn.Embedding` 无对应条目，直接无法查询；
2. **没有超过上限，但超出了训练中充分见过的位置范围**：可以查询，但性能无法保证。

而 Sinusoidal 可以在需要时继续计算更远位置的向量，却同样**不保证模型能够在这些长度上保持预测质量**。

In [ ]:
# 数一下 Learned PE 的额外参数量。
for length in [128, 512, 2048, 8192]:
    params = length * 768
    # fp32 权重本身 4 bytes/参数；训练时还会有梯度和优化器状态。
    weight_mib = params * 4 / 1024**2
    print(f'max_len={length:5d} | 参数={params:>9,d} | 仅 fp32 权重≈{weight_mib:.2f} MiB')

## 9. 为什么有了绝对位置编码，研究者还要提出相对位置编码？

我们仍然使用：

```text
位置 0 与位置 1 → 相隔 1 个 Token
位置 5 与位置 6 → 相隔 1 个 Token
```

从直觉看，两对 Token 的相对关系都可以描述为“后一个 Token 在前一个右侧一格”。

但在可学习绝对位置表中：

$$p_0,p_1,p_5,p_6$$

是通过不同索引独立取出的参数向量。**这种表示没有要求**：

$$\text{Relation}(p_0,p_1)=\text{Relation}(p_5,p_6).$$

换句话说，模型可以通过训练**学习**出相对距离关系，但位置编码机制本身没有把“相隔 1”写成显式的共享规则。

这直接引出下一讲 **Shaw et al. (2018) Relative Position Representations**：

> 能不能让两个 Token 的位置关系直接依赖于它们的距离 $i-j$，而不是仅根据各自的绝对位置向量间接推断？

**注意一个细节：** 上一讲的 Sinusoidal 本身已经通过三角函数带有相对位移的数学结构，因此“绝对位置编码无法表达相对关系”是错误说法。这里真正的区别是：**是否将相对距离显式引入注意力机制**。

## 10. 本讲总结 + 自测

### 你应该掌握的 5 个结论

1. **Learned PE 是一张可训练表。** `nn.Embedding(max_len, d_model)` 里的 `weight` 是形状 `(max_len, d_model)` 的 `nn.Parameter`。
2. **位置 ID 用来索引，而不是直接作为特征值。** ID 3 就取第 3 行，并不意味着它的向量数值必须比位置 2 大。
3. **和 Sinusoidal 一样在 Q/K/V 投影前相加。** 两者都属于典型的绝对位置注入方式。
4. **梯度会更新被使用的位置参数。** 未使用的行在我们采用的普通优化设置下不会发生变化。
5. **绝对位置表并没有显式的距离共享规则。** 这也是后面学习相对位置编码的重要动机之一。

### 3 个自测问题

1. `nn.Embedding(512, 768)` 保存的参数是什么形状？输入 `position_ids.shape=(16,)`，输出是什么形状？
2. 为什么同一个 `苹果` 从位置 0 移动到位置 2 后，Token Embedding 可能相同，但 `Token Embedding + PE` 不同？
3. 为什么把 `max_len` 直接改为 100000，不能保证模型可靠处理 100000 个 Token？

<details>
<summary>展开参考答案</summary>

1. 权重 `(512, 768)`；位置向量 `(16, 768)`。添加 Batch 广播维度可得到 `(1, 16, 768)`。
2. 词 ID 不变时词向量相同，但位置 ID 从 0 变为 2，查到不同的可训练位置向量。
3. 更大的表只新增更多**可查询的参数位置**，不意味着这些参数经历过足够训练；而且模型的长距离注意力等机制同样会影响长上下文性能。

</details>

### 论文与资料

- Vaswani et al. (2017), *Attention Is All You Need*，§3.5 与 Table 3(E)：[NeurIPS 原论文](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)。原文实验发现固定和学习的位置编码表现相近。
- [Google 原始 BERT 实现](https://github.com/google-research/bert/blob/master/modeling.py)：通过 trainable position embedding table 添加绝对位置向量，是另一种实际应用示例；BERT 另外还有 token-type embedding，本文为讲清主线暂时省略。

**下一讲：Shaw Relative Position Encoding（2018）** —— 第一次在我们的学习主线中，把**两个 Token 的相对距离**显式引入 Attention。